# Steepest Descent — Interactive Numerical Experiments

Compare exact-line-search Steepest Descent with Conjugate Gradient
on real SPD and complex Hermitian positive-definite systems.

Explore how matrix conditioning affects convergence and visualize
optimization trajectories on a two-dimensional quadratic surface.

**Instructions:** Select the parameters and click an experiment
button to display its numerical results and plots.


In [ ]:
from pathlib import Path
import sys
import time

# Binder or local Jupyter: find the repository root.
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "gradient_methods").is_dir() and (candidate / "krylov_methods").is_dir():
        root = candidate
        break
else:
    raise RuntimeError("Start Jupyter inside the repository checkout.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.sparse.linalg import cg as scipy_cg

from gradient_methods.steepest_descent_solver import solve_steepest_descent
from krylov_methods.conjugate_gradient_solver import solve_conjugate_gradient
from krylov_methods.notebooks.notebook_utils import (
    spd_problem, hermitian_pd_problem, residual_plot, error_plot, report,
)


In [ ]:
kind = widgets.Dropdown(
    options=[("Real SPD", "real"), ("Complex Hermitian PD", "complex")],
    value="real", description="System",
)
size = widgets.IntSlider(value=20, min=5, max=80, step=5, description="Size n")
condition = widgets.FloatLogSlider(value=1e2, base=10, min=1, max=5,
                                   step=0.5, description="Cond(A)")
tol = widgets.FloatLogSlider(value=1e-8, base=10, min=-11, max=-5,
                             step=1, description="rtol")
budget = widgets.IntSlider(value=2000, min=100, max=8000, step=100,
                           description="Max iterations")
button = widgets.Button(description="Run experiment", button_style="success")
output = widgets.Output()
sd_residual_output = widgets.Output()
sd_error_output = widgets.Output()


def run(_):
    with output:
        clear_output(wait=True)
        sd_residual_output.clear_output(wait=True)
        sd_error_output.clear_output(wait=True)
        A, b, x_true = (
            spd_problem(size.value, condition.value) if kind.value == "real"
            else hermitian_pd_problem(size.value, condition.value)
        )
        threshold = max(1e-12, tol.value * np.linalg.norm(b))
        eigs = np.linalg.eigvalsh(A)
        print("SYSTEM INFORMATION")
        system_label = {"real": "Real SPD", "complex": "Complex Hermitian PD"}[kind.value]
        print(f"Matrix: {system_label}, n={size.value}, dtype={A.dtype}")
        print(f"||A-Aᴴ||₂={np.linalg.norm(A-A.conj().T, ord=2):.3e}")
        print(f"Eigenvalue interval=[{eigs[0]:.3e}, {eigs[-1]:.3e}]")
        print(f"Condition number κ₂(A)={np.linalg.cond(A):.3e}")
        print(f"||b||₂={np.linalg.norm(b):.3e}, ||x*||₂={np.linalg.norm(x_true):.3e}")
        print(f"True residual threshold={threshold:.3e}, budget={budget.value}\n")

        start = time.perf_counter()
        sd = solve_steepest_descent(A, b, rtol=tol.value, atol=1e-12,
                                    max_iter=budget.value, store_history=True)
        sd_time = time.perf_counter() - start
        start = time.perf_counter()
        cg = solve_conjugate_gradient(A, b, rtol=tol.value, atol=1e-12,
                                      max_iter=budget.value, store_history=True)
        cg_time = time.perf_counter() - start

        scipy_hist = [np.linalg.norm(b)]
        scipy_error = [1.0]
        def callback(xk):
            scipy_hist.append(np.linalg.norm(b-A@xk))
            scipy_error.append(np.linalg.norm(xk-x_true)/np.linalg.norm(x_true))
        start = time.perf_counter()
        x_scipy, info = scipy_cg(A, b, rtol=tol.value, atol=1e-12,
                                 maxiter=budget.value, callback=callback)
        scipy_time = time.perf_counter() - start

        report("Steepest Descent", sd.solution, x_true, b, A, sd.iterations, sd.converged)
        report("Our CG", cg.solution, x_true, b, A, cg.iterations, cg.converged)
        report("SciPy CG", x_scipy, x_true, b, A, len(scipy_hist)-1, info == 0)
        print(f"Times (s): SD={sd_time:.4f}, our CG={cg_time:.4f}, SciPy CG={scipy_time:.4f}")
        print(f"SD status: {sd.message}; CG status: {cg.message}; SciPy info={info}")

        histories = {
            "Steepest Descent": sd.residual_norms,
            "Our CG": cg.residual_norms,
            "SciPy CG": scipy_hist,
        }
        error_histories = {
            "Steepest Descent": [
                np.linalg.norm(xk-x_true)/np.linalg.norm(x_true) for xk in sd.history
            ],
            "Our CG": [
                np.linalg.norm(xk-x_true)/np.linalg.norm(x_true) for xk in cg.history
            ],
            "SciPy CG": scipy_error,
        }
        with sd_residual_output:
            residual_plot(histories, threshold, "Steepest Descent vs CG — true residual", figsize=(4.1, 3.0))
        with sd_error_output:
            error_plot(error_histories, "Steepest Descent vs CG — relative solution error", figsize=(4.1, 3.0))
        print("Note: SD decreases the quadratic energy with exact line search, "
              "but the Euclidean residual norm is not guaranteed to decrease each step.")


button.on_click(run)


In [ ]:
path_kappa = widgets.FloatLogSlider(value=1e2, base=10, min=0, max=4,
                                     step=0.5, description="2D Cond(A)")
path_steps = widgets.IntSlider(value=12, min=2, max=40, step=2,
                                description="SD steps")
path_button = widgets.Button(description="Explore optimization path", button_style="info")
path_output = widgets.Output()
path_plot_output = widgets.Output()


def run_path(_):
    with path_output:
        clear_output(wait=True)
        path_plot_output.clear_output(wait=True)
        angle = np.pi / 5
        Q = np.array([[np.cos(angle), -np.sin(angle)],
                      [np.sin(angle), np.cos(angle)]])
        A = Q @ np.diag([1.0, path_kappa.value]) @ Q.T
        x_true = np.array([1.0, 1.0])
        b = A @ x_true
        sd = solve_steepest_descent(A, b, rtol=0.0, atol=0.0,
                                    max_iter=path_steps.value, store_history=True)
        cg = solve_conjugate_gradient(A, b, rtol=1e-14, atol=1e-14,
                                      max_iter=2, store_history=True)
        sd_path = sd.history
        cg_path = cg.history

        xgrid = np.linspace(min(sd_path[:, 0].min(), 0)-0.4, 1.5, 160)
        ygrid = np.linspace(min(sd_path[:, 1].min(), 0)-0.4, 1.5, 160)
        X, Y = np.meshgrid(xgrid, ygrid)
        dx, dy = X-x_true[0], Y-x_true[1]
        energy = 0.5*(A[0,0]*dx**2 + 2*A[0,1]*dx*dy + A[1,1]*dy**2)
        values = energy[energy > 0]
        low = max(np.min(values), 1e-6)
        high = max(np.max(energy), low*10)
        with path_plot_output:
            fig, ax = plt.subplots(figsize=(4.1, 3.0))
            ax.contour(X, Y, energy, levels=np.geomspace(low, high, 14),
                       colors="0.65", linewidths=0.8)
            ax.plot(sd_path[:,0], sd_path[:,1], "o-", ms=3,
                    label="Steepest Descent")
            ax.plot(cg_path[:,0], cg_path[:,1], "s--", ms=5, label="CG")
            ax.scatter([x_true[0]], [x_true[1]], marker="*", s=150,
                       color="black", label="Exact minimizer")
            ax.set(xlabel="$x_1$", ylabel="$x_2$",
                   title=f"Optimization path — condition number {path_kappa.value:.1e}")
            ax.legend()
            ax.grid(alpha=0.2)
            fig.tight_layout()
            plt.show()
        print(f"Steepest Descent iterations={sd.iterations}, "
              f"relative error={np.linalg.norm(sd.solution-x_true)/np.linalg.norm(x_true):.3e}")
        print(f"CG iterations={cg.iterations}, "
              f"relative error={np.linalg.norm(cg.solution-x_true)/np.linalg.norm(x_true):.3e}")


path_button.on_click(run_path)


In [ ]:

# -------------------------------------------------------
# Steepest Descent — Compact Voilà dashboard
# -------------------------------------------------------

sd_status = widgets.HTML()
path_status = widgets.HTML()

sd_parameters = (
    kind, size, condition, tol, budget
)

path_parameters = (
    path_kappa, path_steps
)

last_runs = {
    "sd": None,
    "path": None,
}


# -------------------------------------------------------
# Simulation status indicators
# -------------------------------------------------------

def parameter_values(parameters):
    return tuple(widget.value for widget in parameters)


def update_status(name, parameters, indicator):
    previous = last_runs[name]

    if previous is None:
        indicator.value = ""
    elif parameter_values(parameters) == previous:
        indicator.value = (
            "<span style='color:#22863a;font-weight:600;'>"
            "● Simulation up to date</span>"
        )
    else:
        indicator.value = (
            "<span style='color:#b36b00;font-weight:600;'>"
            "● Parameters changed — rerun required</span>"
        )


def refresh_sd(change=None):
    update_status("sd", sd_parameters, sd_status)


def refresh_path(change=None):
    update_status("path", path_parameters, path_status)


for parameter in sd_parameters:
    parameter.observe(refresh_sd, names="value")

for parameter in path_parameters:
    parameter.observe(refresh_path, names="value")


# -------------------------------------------------------
# Three independent plot panels
# -------------------------------------------------------

def plot_panel(title, plot_output):
    return widgets.VBox(
        [
            widgets.HTML(
                "<div style='font-weight:600;"
                "text-align:center;font-size:13px;'>"
                + title
                + "</div>"
            ),
            plot_output,
        ],
        layout=widgets.Layout(
            flex="1 1 280px",
            min_width="240px",
            max_width="100%",
            padding="4px",
            overflow="hidden",
        ),
    )


sd_residual_panel = plot_panel(
    "True Residual Convergence",
    sd_residual_output,
)

sd_error_panel = plot_panel(
    "Relative Solution Error",
    sd_error_output,
)

path_panel = plot_panel(
    "Optimization Trajectories",
    path_plot_output,
)

all_plot_panels = (
    sd_residual_panel,
    sd_error_panel,
    path_panel,
)

# Hide every plot before its experiment runs.
for panel in all_plot_panels:
    panel.layout.display = "none"

plots_row = widgets.HBox(
    list(all_plot_panels),
    layout=widgets.Layout(
        width="100%",
        flex_flow="row wrap",
        gap="8px",
        align_items="flex-start",
    ),
)


# -------------------------------------------------------
# Expandable numerical reports
# -------------------------------------------------------

sd_details = widgets.Accordion(
    children=[output]
)

sd_details.set_title(
    0, "Steepest Descent & CG solver comparison"
)

sd_details.selected_index = None


path_details = widgets.Accordion(
    children=[path_output]
)

path_details.set_title(
    0, "Optimization-path numerical details"
)

path_details.selected_index = None


for details in (sd_details, path_details):
    details.layout.display = "none"
    details.layout.flex = "1 1 400px"
    details.layout.min_width = "260px"


# -------------------------------------------------------
# Reveal results after experiment execution
# -------------------------------------------------------

def show_sd_results(change=None):
    sd_residual_panel.layout.display = ""
    sd_error_panel.layout.display = ""
    sd_details.layout.display = ""

    last_runs["sd"] = parameter_values(
        sd_parameters
    )

    refresh_sd()


def show_path_results(change=None):
    path_panel.layout.display = ""
    path_details.layout.display = ""

    last_runs["path"] = parameter_values(
        path_parameters
    )

    refresh_path()


# Register display callbacks after existing
# numerical callbacks, preserving the algorithms.
button.on_click(show_sd_results)
path_button.on_click(show_path_results)


# -------------------------------------------------------
# Experiment controls
# -------------------------------------------------------

sd_controls = widgets.VBox(
    [
        widgets.HTML(
            "<h3>Steepest Descent vs CG</h3>"
        ),
        kind,
        size,
        condition,
        tol,
        budget,
        button,
        sd_status,
    ],
    layout=widgets.Layout(
        flex="1 1 450px",
        min_width="300px",
        padding="10px",
        border="1px solid #dddddd",
    ),
)

path_controls = widgets.VBox(
    [
        widgets.HTML(
            "<h3>Optimization Path</h3>"
        ),
        widgets.HTML(
            "<p style='font-size:13px;'>"
            "Explore how conditioning changes "
            "the optimization trajectory on a "
            "two-dimensional quadratic surface."
            "</p>"
        ),
        path_kappa,
        path_steps,
        path_button,
        path_status,
    ],
    layout=widgets.Layout(
        flex="1 1 450px",
        min_width="280px",
        padding="10px",
        border="1px solid #dddddd",
    ),
)

controls_row = widgets.HBox(
    [
        sd_controls,
        path_controls,
    ],
    layout=widgets.Layout(
        width="100%",
        flex_flow="row wrap",
        gap="10px",
        align_items="stretch",
    ),
)

details_row = widgets.HBox(
    [
        sd_details,
        path_details,
    ],
    layout=widgets.Layout(
        width="100%",
        flex_flow="row wrap",
        gap="10px",
        align_items="flex-start",
    ),
)


# -------------------------------------------------------
# Final interactive dashboard
# -------------------------------------------------------

steepest_descent_dashboard = widgets.VBox(
    [
        widgets.HTML("""
        <style>
        .sd-compact-dashboard .output_area img,
        .sd-compact-dashboard .jp-OutputArea-output img {
            max-width: 100% !important;
            height: auto !important;
        }
        </style>
        """),
        controls_row,
        plots_row,
        details_row,
    ],
    layout=widgets.Layout(
        width="100%",
        gap="12px",
    ),
)

steepest_descent_dashboard.add_class(
    "sd-compact-dashboard"
)

display(steepest_descent_dashboard)

# No automatic simulations.
# Plots and numerical reports appear only after
# their corresponding experiment buttons are clicked.
